# 02 – Batch processing et optimisation

Ce notebook réalise l'analyse batch des données Mastodon stockées dans PostgreSQL avec Apache Spark.

Les objectifs sont :
- charger les données historiques depuis PostgreSQL ;
- analyser l'activité des utilisateurs ;
- calculer le nombre de toots par jour ;
- identifier les hashtags les plus fréquents ;
- calculer des statistiques sur la longueur des toots ;
- tester les optimisations Spark avec cache, repartition et coalesce.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import time

spark = (
    SparkSession.builder
    .appName("02-mastodon-batch")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Version Spark :", spark.version)
print("Application :", spark.sparkContext.appName)
print("Spark UI :", spark.sparkContext.uiWebUrl)
print("Parallélisme par défaut :", spark.sparkContext.defaultParallelism)

D:\mastodon-spark-project\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Version Spark : 4.2.0
Application : 02-mastodon-batch
Spark UI : http://MSI.home:4040
Parallélisme par défaut : 16


## 1. Chargement des données depuis PostgreSQL

Les données Mastodon collectées sont stockées dans PostgreSQL.

Cette étape utilise JDBC pour charger la table `toots` dans un DataFrame Spark afin de réaliser les traitements batch.

In [2]:
jdbc_url = "jdbc:postgresql://localhost:5433/mastodon"

jdbc_properties = {
    "user": "spark",
    "password": "spark123",
    "driver": "org.postgresql.Driver"
}

toots = spark.read.jdbc(
    url=jdbc_url,
    table="toots",
    properties=jdbc_properties
)

print("Nombre de toots :", toots.count())

Nombre de toots : 5


In [3]:
toots.printSchema()

root
 |-- id: long (nullable = true)
 |-- toot_id: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- user_id: string (nullable = true)
 |-- username: string (nullable = true)
 |-- content: string (nullable = true)
 |-- language: string (nullable = true)
 |-- hashtags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- favourites_count: integer (nullable = true)
 |-- reblogs_count: integer (nullable = true)



In [4]:
toots.show(10, truncate=False)

+---+-------+--------------------------+-------+--------+---------------------------------------+--------+---------------------------+----------------+-------------+
|id |toot_id|created_at                |user_id|username|content                                |language|hashtags                   |favourites_count|reblogs_count|
+---+-------+--------------------------+-------+--------+---------------------------------------+--------+---------------------------+----------------+-------------+
|1  |1001   |2026-09-24 09:36:18.154775|u1     |alice   |I love Apache Spark and Data Science   |en      |[Spark, DataScience]       |12              |3            |
|2  |1002   |2026-09-24 08:36:18.154775|u2     |bob     |Learning Kafka with Spark streaming    |en      |[Kafka, Spark]             |8               |2            |
|3  |1003   |2026-09-23 09:36:18.154775|u1     |alice   |AI and machine learning are interesting|en      |[AI, MachineLearning]      |20              |5            |
|4  

In [5]:
toots = toots.withColumn(
    "text_length",
    F.length("content")
)

toots.select(
    "toot_id",
    "username",
    "created_at",
    "hashtags",
    "content",
    "text_length"
).show(10, truncate=False)

+-------+--------+--------------------------+---------------------------+---------------------------------------+-----------+
|toot_id|username|created_at                |hashtags                   |content                                |text_length|
+-------+--------+--------------------------+---------------------------+---------------------------------------+-----------+
|1001   |alice   |2026-09-24 09:36:18.154775|[Spark, DataScience]       |I love Apache Spark and Data Science   |36         |
|1002   |bob     |2026-09-24 08:36:18.154775|[Kafka, Spark]             |Learning Kafka with Spark streaming    |35         |
|1003   |alice   |2026-09-23 09:36:18.154775|[AI, MachineLearning]      |AI and machine learning are interesting|39         |
|1004   |charlie |2026-09-23 09:36:18.154775|[PostgreSQL, Spark]        |PostgreSQL works great with Spark      |33         |
|1005   |bob     |2026-09-22 09:36:18.154775|[DataEngineering, Mastodon]|Data engineering project with Mastodon |38   

### Préparation des données

La colonne `text_length` contient le nombre de caractères de chaque toot.

Les hashtags sont déjà stockés sous forme de tableau dans PostgreSQL, ils peuvent donc être directement utilisés avec les fonctions Spark comme `explode()`.

## 2. Transformations et agrégations

Nous analysons maintenant les données historiques Mastodon afin de :

- identifier les utilisateurs les plus actifs ;
- compter le nombre de toots par jour ;
- trouver les hashtags les plus fréquents ;
- calculer la longueur moyenne des toots.

In [6]:
# Activité par utilisateur
user_activity = (
    toots
    .groupBy("user_id", "username")
    .agg(
        F.count("*").alias("nb_toots")
    )
    .orderBy(F.desc("nb_toots"))
)

user_activity.show()

+-------+--------+--------+
|user_id|username|nb_toots|
+-------+--------+--------+
|     u2|     bob|       2|
|     u1|   alice|       2|
|     u3| charlie|       1|
+-------+--------+--------+



In [7]:
# Pour les données de test, on considère actif un utilisateur
# ayant publié plus d'un toot.
active_users = (
    user_activity
    .filter(F.col("nb_toots") > 1)
)

print("Nombre d'utilisateurs actifs :", active_users.count())
active_users.show()

Nombre d'utilisateurs actifs : 2
+-------+--------+--------+
|user_id|username|nb_toots|
+-------+--------+--------+
|     u2|     bob|       2|
|     u1|   alice|       2|
+-------+--------+--------+



### Nombre de toots par jour

La colonne `created_at` est transformée en date afin de compter le nombre total de publications pour chaque journée.

In [8]:
toots_per_day = (
    toots
    .withColumn("day", F.to_date("created_at"))
    .groupBy("day")
    .agg(
        F.count("*").alias("nb_toots")
    )
    .orderBy("day")
)

toots_per_day.show()

+----------+--------+
|       day|nb_toots|
+----------+--------+
|2026-09-22|       1|
|2026-09-23|       2|
|2026-09-24|       2|
+----------+--------+



### Hashtags les plus fréquents

La fonction `explode()` transforme chaque élément du tableau `hashtags`
en une ligne distincte. Nous pouvons ensuite compter les occurrences
de chaque hashtag.

In [9]:
top_hashtags = (
    toots
    .select(
        F.explode("hashtags").alias("hashtag")
    )
    .filter(
        F.col("hashtag").isNotNull()
        & (F.trim(F.col("hashtag")) != "")
    )
    .groupBy("hashtag")
    .agg(
        F.count("*").alias("nb_occurrences")
    )
    .orderBy(F.desc("nb_occurrences"))
)

top_hashtags.show(10, truncate=False)

+---------------+--------------+
|hashtag        |nb_occurrences|
+---------------+--------------+
|Spark          |3             |
|PostgreSQL     |1             |
|Mastodon       |1             |
|AI             |1             |
|Kafka          |1             |
|MachineLearning|1             |
|DataEngineering|1             |
|DataScience    |1             |
+---------------+--------------+



In [10]:
average_length = (
    toots
    .agg(
        F.avg("text_length").alias("avg_text_length")
    )
)

average_length.show()

+---------------+
|avg_text_length|
+---------------+
|           36.2|
+---------------+



In [11]:
average_length_by_user = (
    toots
    .groupBy("username")
    .agg(
        F.count("*").alias("nb_toots"),
        F.avg("text_length").alias("avg_text_length")
    )
    .orderBy(F.desc("avg_text_length"))
)

average_length_by_user.show()

+--------+--------+---------------+
|username|nb_toots|avg_text_length|
+--------+--------+---------------+
|   alice|       2|           37.5|
|     bob|       2|           36.5|
| charlie|       1|           33.0|
+--------+--------+---------------+



## 3. Analyse avec Spark SQL

Spark permet d'interroger un DataFrame avec une syntaxe SQL.

Nous créons une vue temporaire à partir du DataFrame `toots`, puis nous
reproduisons certaines analyses précédentes avec Spark SQL.

In [12]:
toots.createOrReplaceTempView("toots")

print("Vue SQL 'toots' créée.")

Vue SQL 'toots' créée.


3.1 Nombre de toots par jour avec SQL

In [13]:
daily_sql = spark.sql("""
    SELECT
        TO_DATE(created_at) AS day,
        COUNT(*) AS nb_toots
    FROM toots
    GROUP BY TO_DATE(created_at)
    ORDER BY day
""")

daily_sql.show()

+----------+--------+
|       day|nb_toots|
+----------+--------+
|2026-09-22|       1|
|2026-09-23|       2|
|2026-09-24|       2|
+----------+--------+



In [14]:
daily_sql.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [day#211 ASC NULLS FIRST], true, 0
   +- Exchange rangepartitioning(day#211 ASC NULLS FIRST, 200), ENSURE_REQUIREMENTS, [plan_id=542]
      +- HashAggregate(keys=[_groupingexpression#224], functions=[count(1)])
         +- Exchange hashpartitioning(_groupingexpression#224, 200), ENSURE_REQUIREMENTS, [plan_id=539]
            +- HashAggregate(keys=[_groupingexpression#224], functions=[partial_count(1)])
               +- Project [cast(created_at#2 as date) AS _groupingexpression#224]
                  +- Scan JDBCRelation(toots) [numPartitions=1] [created_at#2] PushedFilters: [], ReadSchema: struct<created_at:timestamp>




### Observation

Cette requête SQL produit le même résultat que l'API DataFrame utilisée précédemment.

`explain()` permet d'observer le plan d'exécution choisi par Spark pour
réaliser les filtres, agrégations et tris.

3.2 Utilisateurs actifs avec SQL

In [15]:
active_users_sql = spark.sql("""
    SELECT
        user_id,
        username,
        COUNT(*) AS nb_toots
    FROM toots
    GROUP BY user_id, username
    HAVING COUNT(*) > 1
    ORDER BY nb_toots DESC
""")

active_users_sql.show()

+-------+--------+--------+
|user_id|username|nb_toots|
+-------+--------+--------+
|     u2|     bob|       2|
|     u1|   alice|       2|
+-------+--------+--------+



In [16]:
active_users_sql.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [nb_toots#225L DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(nb_toots#225L DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=626]
      +- Filter (nb_toots#225L > 1)
         +- HashAggregate(keys=[user_id#3, username#4], functions=[count(1)])
            +- Exchange hashpartitioning(user_id#3, username#4, 200), ENSURE_REQUIREMENTS, [plan_id=622]
               +- HashAggregate(keys=[user_id#3, username#4], functions=[partial_count(1)])
                  +- Scan JDBCRelation(toots) [numPartitions=1] [user_id#3,username#4] PushedFilters: [], ReadSchema: struct<user_id:string,username:string>




3.3 Top hashtags avec SQL

In [17]:
hashtags_sql = spark.sql("""
    SELECT
        hashtag,
        COUNT(*) AS nb_occurrences
    FROM (
        SELECT EXPLODE(hashtags) AS hashtag
        FROM toots
    )
    WHERE hashtag IS NOT NULL
      AND TRIM(hashtag) <> ''
    GROUP BY hashtag
    ORDER BY nb_occurrences DESC
""")

hashtags_sql.show(10, truncate=False)

+---------------+--------------+
|hashtag        |nb_occurrences|
+---------------+--------------+
|Spark          |3             |
|PostgreSQL     |1             |
|Mastodon       |1             |
|AI             |1             |
|Kafka          |1             |
|MachineLearning|1             |
|DataEngineering|1             |
|DataScience    |1             |
+---------------+--------------+



In [18]:
print("DataFrame API :")
toots_per_day.show()

print("Spark SQL :")
daily_sql.show()

DataFrame API :
+----------+--------+
|       day|nb_toots|
+----------+--------+
|2026-09-22|       1|
|2026-09-23|       2|
|2026-09-24|       2|
+----------+--------+

Spark SQL :
+----------+--------+
|       day|nb_toots|
+----------+--------+
|2026-09-22|       1|
|2026-09-23|       2|
|2026-09-24|       2|
+----------+--------+



## 4. Optimisation Spark

Nous comparons plusieurs configurations afin d'observer l'effet du cache
et du partitionnement sur les traitements batch.

Les tests portent sur :
- une lecture PostgreSQL sans cache ;
- un DataFrame mis en cache ;
- un DataFrame repartitionné ;
- une réduction du nombre de partitions avec `coalesce`.

In [19]:
def run_workload(df):
    # Nombre de toots par jour
    result_day = (
        df
        .withColumn("day", F.to_date("created_at"))
        .groupBy("day")
        .agg(F.count("*").alias("nb_toots"))
        .orderBy("day")
    )

    # Top hashtags
    result_hashtags = (
        df
        .select(F.explode("hashtags").alias("hashtag"))
        .filter(
            F.col("hashtag").isNotNull()
            & (F.trim(F.col("hashtag")) != "")
        )
        .groupBy("hashtag")
        .agg(F.count("*").alias("nb_occurrences"))
        .orderBy(F.desc("nb_occurrences"))
    )

    # Longueur moyenne par langue
    result_language = (
        df
        .groupBy("language")
        .agg(
            F.avg("text_length").alias("avg_text_length")
        )
    )

    # Actions pour forcer réellement l'exécution Spark
    result_day.collect()
    result_hashtags.collect()
    result_language.collect()

In [20]:
def measure_workload(name, df):
    start = time.time()

    run_workload(df)

    duration = time.time() - start
    partitions = df.rdd.getNumPartitions()

    print(name)
    print(f"Durée : {duration:.4f} secondes")
    print(f"Nombre de partitions : {partitions}")
    print()

    return duration, partitions

Teste sans cache

In [21]:
toots.unpersist(blocking=True)
spark.catalog.clearCache()

duration_no_cache, partitions_no_cache = measure_workload(
    "PostgreSQL sans cache",
    toots
)

PostgreSQL sans cache
Durée : 1.8129 secondes
Nombre de partitions : 1



Teste avec cache

In [22]:
toots_cached = toots.cache()

start = time.time()
toots_cached.count()
duration_materialization = time.time() - start

print(
    f"Durée de matérialisation du cache : "
    f"{duration_materialization:.4f} secondes"
)

Durée de matérialisation du cache : 0.5285 secondes


In [23]:
duration_cache, partitions_cache = measure_workload(
    "Cache matérialisé",
    toots_cached
)

Cache matérialisé
Durée : 1.5891 secondes
Nombre de partitions : 1



Repartition

In [24]:
toots_cached.unpersist(blocking=True)

toots_repartitioned = (
    toots
    .repartition(4, "language")
    .cache()
)

toots_repartitioned.count()

print(
    "Partitions après repartition :",
    toots_repartitioned.rdd.getNumPartitions()
)

Partitions après repartition : 4


In [25]:
duration_repartition, partitions_repartition = measure_workload(
    "Repartition(4, language) + cache",
    toots_repartitioned
)

Repartition(4, language) + cache
Durée : 2.0042 secondes
Nombre de partitions : 4



Coalesce

In [26]:
toots_coalesced = toots_repartitioned.coalesce(2)

duration_coalesce, partitions_coalesce = measure_workload(
    "Coalesce(2)",
    toots_coalesced
)

Coalesce(2)
Durée : 1.5777 secondes
Nombre de partitions : 2



In [27]:
optimization_results = spark.createDataFrame(
    [
        ("PostgreSQL sans cache", duration_no_cache, partitions_no_cache),
        ("Cache matérialisé", duration_cache, partitions_cache),
        (
            "Repartition(4, language) + cache",
            duration_repartition,
            partitions_repartition
        ),
        ("Coalesce(2)", duration_coalesce, partitions_coalesce),
    ],
    ["configuration", "duree_secondes", "nb_partitions"]
)

optimization_results.show(truncate=False)

+--------------------------------+------------------+-------------+
|configuration                   |duree_secondes    |nb_partitions|
+--------------------------------+------------------+-------------+
|PostgreSQL sans cache           |1.8129448890686035|1            |
|Cache matérialisé               |1.5891375541687012|1            |
|Repartition(4, language) + cache|2.0042076110839844|4            |
|Coalesce(2)                     |1.5776727199554443|2            |
+--------------------------------+------------------+-------------+



### Interprétation

Les mesures doivent être interprétées avec prudence car le jeu de données
de test contient actuellement très peu de lignes.

Le cache devient utile lorsqu'un même DataFrame est réutilisé plusieurs fois.

`repartition()` provoque une redistribution complète des données afin de
modifier ou rééquilibrer les partitions.

`coalesce()` est principalement utilisé pour réduire le nombre de partitions
en évitant généralement un shuffle complet.

Les différences de performance seront plus représentatives lorsque la base
contiendra davantage de données Mastodon.

## 5. Spark UI et nettoyage

Spark UI permet d'observer les jobs, les stages, les shuffles,
les partitions et les données mises en cache.

Dans notre environnement, l'interface est disponible à l'adresse
indiquée par `spark.sparkContext.uiWebUrl`.

Les mesures actuelles utilisent seulement quelques données de test ;
elles devront être refaites avec un volume Mastodon plus important.

In [28]:
print("Spark UI :", spark.sparkContext.uiWebUrl)

toots.unpersist(blocking=True)
toots_cached.unpersist(blocking=True)
toots_repartitioned.unpersist(blocking=True)

spark.catalog.clearCache()

print("Caches Spark libérés.")

Spark UI : http://MSI.home:4040
Caches Spark libérés.


## Conclusion

Le traitement batch permet de charger les données Mastodon depuis PostgreSQL
avec Spark JDBC puis d'effectuer plusieurs analyses :

- activité des utilisateurs ;
- nombre de toots par jour ;
- hashtags les plus fréquents ;
- longueur moyenne des publications ;
- requêtes équivalentes avec Spark SQL ;
- comparaison de plusieurs techniques d'optimisation Spark.

Les tests de performance avec `cache`, `repartition` et `coalesce`
seront plus représentatifs avec les données réellement collectées
par le pipeline de streaming Mastodon.